In [ ]:
from pysqlite3 import dbapi2 as sqlite3
import aloelite, re
from aloelite import Db, NodeId, NodeType, errors, resolve, resolve_parent
from aloelite.models import NodeInfo, VolumeInfo

TEMPLATES_FILE = "config/sql-templates.yaml"
SCHEMA_FILE = "sql/schema.sql"

db = Db.open(":memory:", TEMPLATES_FILE, schema_path=SCHEMA_FILE)
UUID7 = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-7[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"
)

with db.txn():
    vid = db.gen_id()
    db.run("mutation.create_volume", {"id": vid, "name": "v"})
    root = db.create_monotonic(
        "mutation.create_node",
        "mutation.get_generated_node_id",
        {
            "type": "container",
            "name": "/",
            "created_at": None,
            "modified_at": None,
            "volume": vid,
        },
    )
    db.rowcount("mutation.link_root", {"volume": vid, "root": root})
assert UUID7.match(vid) and UUID7.match(root)
print("bootstrap: volume(stateless id)+root(monotonic id) in one txn OK")

vinfo = VolumeInfo.from_row(db.one("resolution.get_volume", {"volume": vid}))
assert vinfo.api_version == 1 and vinfo.root == root
print("VolumeInfo.from_row OK (api_version=1, root linked)")


def mkc(parent, name):
    with db.txn():
        nid = db.create_monotonic(
            "mutation.create_node",
            "mutation.get_generated_node_id",
            {
                "type": "container",
                "name": name,
                "created_at": None,
                "modified_at": None,
                "volume": vid,
            },
        )
        db.create_monotonic(
            "mutation.create_edge",
            "mutation.get_generated_edge_id",
            {"from_id": parent, "to_id": nid, "volume": vid},
        )
    return NodeId(nid)


def mke(parent, name):
    with db.txn():
        nid = db.create_monotonic(
            "mutation.create_node",
            "mutation.get_generated_node_id",
            {
                "type": "entry",
                "name": name,
                "created_at": None,
                "modified_at": None,
                "volume": vid,
            },
        )
        db.run("mutation.create_content", {"node": nid, "payload": b"", "hash": None})
        db.create_monotonic(
            "mutation.create_edge",
            "mutation.get_generated_edge_id",
            {"from_id": parent, "to_id": nid, "volume": vid},
        )
    return NodeId(nid)


a = mkc(root, "a")
b = mkc(a, "b")
c = mke(b, "c")
x = mke(a, "x")

assert resolve(db, NodeId(root), "/a/b/c").node == c
assert (
    resolve(db, NodeId(root), "/").node == root
    and resolve(db, NodeId(root), "").node == root
)
assert resolve(db, NodeId(root), "/a//b/").node == b
print("resolve: full path, root, '', collapse OK")

for bad, exc in [("/a/nope", errors.NotFound), ("/a/x/deep", errors.NotAContainer)]:
    try:
        resolve(db, NodeId(root), bad)
        raise SystemExit("FAIL " + bad)
    except exc:
        pass
print("resolve: NotFound + NotAContainer OK")

p = resolve_parent(db, NodeId(root), "/a/b/new")
assert p == (b, "new")
try:
    resolve_parent(db, NodeId(root), "/")
    raise SystemExit("FAIL")
except errors.NotFound:
    pass
print("resolve_parent OK")

# NODE-5 newest wins
with db.txn():
    x2 = db.create_monotonic(
        "mutation.create_node",
        "mutation.get_generated_node_id",
        {
            "type": "entry",
            "name": "x",
            "created_at": None,
            "modified_at": None,
            "volume": vid,
        },
    )
    db.run("mutation.create_content", {"node": x2, "payload": b"", "hash": None})
    db.create_monotonic(
        "mutation.create_edge",
        "mutation.get_generated_edge_id",
        {"from_id": a, "to_id": x2, "volume": vid},
    )
assert resolve(db, NodeId(root), "/a/x").node == x2
print("NODE-5: newest same-name wins OK")

# txn rollback leaves no trace
try:
    with db.txn():
        db.create_monotonic(
            "mutation.create_node",
            "mutation.get_generated_node_id",
            {
                "type": "container",
                "name": "ghost",
                "created_at": None,
                "modified_at": None,
                "volume": vid,
            },
        )
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert all(
    r["name"] != "ghost"
    for r in db.all("resolution.list_children", {"container": root})
)
print("txn rollback: no trace OK")

ninfo = NodeInfo.from_row(db.one("resolution.get_node", {"node": c}))
assert ninfo.modified_at == ninfo.created_at and ninfo.type is NodeType.ENTRY
print("NodeInfo.from_row: fresh modified_at==created_at OK")

# error registry mirrors closed enum
assert errors.BY_CODE["would_cycle"] is errors.WouldCycle
print("error registry by code OK")
print("\nALL ALOEFS FOUNDATION CHECKS PASSED")

In [ ]:
"""
Aloelite — getting started
==========================
Run:  python3 getting_started.py
Requires:  pip install cryptography pydantic pyyaml msgpack
"""

import os
import sys

# If running from the repo root (aloelite/ package is a sibling directory),
# nothing extra is needed. If you're running from inside the package, add the
# parent to sys.path:
#   sys.path.insert(0, "..")

from aloelite.aloelite import Aloelite
from aloelite.types import WriteMode, Whence

DB = "aloelite_test.sqlite"
ENC_DB = "aloelite_enc_test.sqlite"
for f in (DB, ENC_DB):
    if os.path.exists(f):
        os.unlink(f)


# ===========================================================================
# 1. Basics (unencrypted volume)
# ===========================================================================
print("=" * 60)
print("1. Basics")
print("=" * 60)

with Aloelite(DB) as fs:  # owns the file (WAL); closes on exit
    vol = fs.create_volume("photos")  # a volume = an origin / namespace
    print(fs.list_volumes())  # -> [VolumeInfo(...)]

    with fs.mount(vol.id) as m:  # a session; unmounts on exit
        # --- create & write -------------------------------------------------
        m.create_container("/2024")
        m.set_metadata(
            "/2024", {"year": "2024", "album": "trip"}
        )  # shallow {str:str}, any node
        m.create_entry("/2024/caption.txt", b"a sunset")  # create with data
        m.set_metadata("/2024/caption.txt", {"alt": "a sunset"})

        with m.open_write("/note.txt") as w:  # create-on-write (truncate mode)
            w.write(b"hello ")
            w.write(b"world")
        m.write_all("/note.txt", b"replaced")  # atomic full replace

        # --- append --------------------------------------------------------
        with m.open_write("/note.txt", WriteMode.APPEND) as w:  # must already exist
            w.write(b" + more")

        # --- read ----------------------------------------------------------
        print(m.read_all("/note.txt"))  # -> b"replaced + more"

        with m.open_read("/note.txt") as r:
            head = r.read(8)  # streaming ranged read
            r.seek(-4, Whence.END)
            tail = r.read()
        print(f"  head={head!r}  tail={tail!r}")

        # --- browse & inspect ----------------------------------------------
        for e in m.list("/2024"):
            print(e.name, e.type.value, e.visible)

        info = m.stat(
            "/2024/caption.txt"
        )  # NodeInfo: id, type, size, created_at, modified_at
        print(m.exists("/nope"))  # -> False
        print(m.path_of(info.id))  # id -> "/2024/caption.txt"

        n = m.stat("/note.txt")
        print(f"note.txt size: {n.size} bytes")

        # --- move / rename / copy ------------------------------------------
        m.rename("/note.txt", "readme.txt")
        m.move("/readme.txt", "/2024/readme.txt")  # dst is the full path
        m.copy("/2024", "/backup")  # deep copy, fresh ids

        # --- metadata ------------------------------------------------------
        print(m.stat("/2024").metadata)  # -> {'album': 'trip', 'year': '2024'}
        print(m.stat("/backup").metadata)  # -> same map; copy preserved it (OP-4r1)
        m.set_metadata("/2024", {})  # whole-map replace; {} clears

        # --- delete --------------------------------------------------------
        m.remove("/2024/readme.txt")  # leaf / empty container only
        m.remove_recursive("/backup")  # whole subtree

        # --- pack a subtree into one entry, then restore -------------------
        packed = m.pack("/2024")  # /2024 becomes a packed entry
        m.unpack("/2024")  # restored from the blob

    fs.prune()  # reclaim orphans + dead locks
    print(fs.health_check())  # -> [] when consistent


# ===========================================================================
# 2. Encrypted volume
#    At-rest guarantee: the SQLite file is opaque without the PIN.
#    Dedup is preserved in convergent mode (default): identical chunks store
#    once and the encrypted block is identical too, so the pool still collapses
#    duplicates. Use enc_mode="random" to trade dedup for zero equality leakage.
# ===========================================================================
print()
print("=" * 60)
print("2. Encryption")
print("=" * 60)

PIN = b"correct-horse-battery-staple"

with Aloelite(ENC_DB) as fs:
    # create_volume(pin=...) provisions a random volume key K_v sealed under
    # Argon2id(PIN). The PIN is never stored; the key is.
    vol = fs.create_volume("vault", pin=PIN)  # enc_mode="convergent" (default)
    # vol = fs.create_volume("vault", pin=PIN, enc_mode="random")  # no dedup, no equality leakage

    # mount(pin=...) derives K_u from the PIN, unwraps K_v, installs the chunk
    # cipher, and returns a per-session token (Mount.token). The token + the
    # stored mount nonce N_m let you reconstruct K_v within the same process
    # without re-entering the PIN (useful for a cache layer or worker handoff).
    with fs.mount(vol.id, pin=PIN) as m:
        print(f"session token (runtime-only): {m.token.hex()}")

        # All write/read operations are identical to the unencrypted case —
        # encryption is invisible at the Mount API level.
        m.create_entry("/secret.txt", b"eyes only")
        print(m.read_all("/secret.txt"))  # -> b"eyes only"

        # Streaming writes are also encrypted chunk-by-chunk (bounded memory
        # is preserved: peak pending < chunk_size regardless of file size).
        with m.open_write("/big.bin") as w:
            for i in range(20):
                w.write(bytes([i % 251] * 4096))
        print(f"big.bin size: {m.stat('/big.bin').size} bytes")

    # After unmount the cipher is cleared; the file is opaque without the PIN.

    # Re-open with the correct PIN.
    with fs.mount(vol.id, pin=PIN) as m:
        print(m.read_all("/secret.txt"))  # -> b"eyes only"

    # Wrong PIN is rejected at mount time (not at read time).
    from aloelite import errors

    try:
        fs.mount(vol.id, pin=b"wrong")
    except errors.BadKey:
        print("wrong PIN rejected at mount  ✓")

    # PIN mismatch errors surface immediately (pin on a plain volume, or none
    # on an encrypted one).
    plain_vol = fs.create_volume("plain")  # no pin -> enc_mode 'none'
    try:
        fs.mount(plain_vol.id, pin=PIN)
    except errors.EncryptionRequired:
        print("pin on plain volume rejected  ✓")

    try:
        fs.mount(vol.id)  # encrypted volume, no pin given
    except errors.EncryptionRequired:
        print("no pin on encrypted volume rejected  ✓")

1. Basics
[VolumeInfo(id='019efc2f-5543-79d2-ae8f-ebfeee54a2e9', name='photos', root='019efc2f-5543-7000-b90f-22a4684f0efa', api_version=1, created_at=1782347420995)]
b'replaced + more'
  head=b'replaced'  tail=b'more'
caption.txt entry True
False
/2024/caption.txt
note.txt size: 15 bytes
{'album': 'trip', 'year': '2024'}
{'album': 'trip', 'year': '2024'}
[]

2. Encryption
session token (runtime-only): 1d34c02de7c7ac184143153cfc6dd961
b'eyes only'
big.bin size: 81920 bytes
b'eyes only'
wrong PIN rejected at mount  ✓
pin on plain volume rejected  ✓
no pin on encrypted volume rejected  ✓


In [ ]:
with Aloelite("photos.sqlite") as fs:
    vid = next(v.id for v in fs.list_volumes() if v.name == "photos")
    with fs.mount(vid) as m:
        m.create_entry("/sz.txt", b"")
        m.write_all("/sz.txt", b"hello world")  # 11 bytes
        info = m.stat("/sz.txt")
        by_id = m.stat_by_id(info.id)
        print("stat.size   =", info.size)
        print("by_id.size  =", by_id.size)

stat.size   = 11
by_id.size  = 11
